In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import os

# ==========================================
# Parameters
# ==========================================

INPUT_FILE = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\time_scale2.csv"

OUTPUT_DIR = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_Lag_DS_interpolate"

PSEUDOCOUNT = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# Lag & Interpolation parameters
# ==========================================

# "DS"  : DS领先，研究 DS(t) -> ADS(t + lag)
# "ADS" : ADS领先，研究 ADS(t) -> DS(t + lag)
INDEPENDENT_VARIABLE = "DS"

# 需要测试的精确滞后天数
TARGET_LAGS = [0, 7, 10, 14, 21, 30]

# 两个真实采样点之间，如果间隔超过这个值，则不进行线性插值
INTERPOLATE_LIMIT = 60

# 最小有效配对数
MIN_N = 5


# ==========================================
# 1. Load & Clean data
# ==========================================

print("Loading data...")

df = pd.read_csv(INPUT_FILE)

# 删除完全为空的列
df = df.dropna(axis=1, how="all")

# 只保留前7列
df = df.iloc[:, :7]

df.columns = [
    "Environment",
    "Sample_ID",
    "Time",
    "DS",
    "DS_TPM",
    "ADS",
    "ADS_TPM"
]

# TPM转为数值
df["DS_TPM"] = pd.to_numeric(df["DS_TPM"], errors="coerce")
df["ADS_TPM"] = pd.to_numeric(df["ADS_TPM"], errors="coerce")


# ==========================================
# 2. Parse time
# ==========================================

def parse_time(x):

    x = str(x).strip()

    # YYYYMM
    if len(x) == 6 and x.isdigit():
        return pd.to_datetime(
            x,
            format="%Y%m",
            errors="coerce"
        )

    # 普通日期
    return pd.to_datetime(
        x,
        errors="coerce"
    )


df["Time_parsed"] = df["Time"].apply(parse_time)

df = df.dropna(
    subset=["Time_parsed"]
)


# ==========================================
# 3. Aggregate repeated observations
# ==========================================

print("Aggregating repeated observations...")

ds_summary = (
    df.groupby(
        ["Environment", "Time_parsed", "DS"],
        as_index=False
    )["DS_TPM"]
    .mean()
)

ads_summary = (
    df.groupby(
        ["Environment", "Time_parsed", "ADS"],
        as_index=False
    )["ADS_TPM"]
    .mean()
)


# ==========================================
# 4. CLR transformation
# ==========================================

def clr_transform(series, pseudocount=1):

    x = series + pseudocount

    log_x = np.log(x)

    return log_x - log_x.mean()


ds_summary["DS_CLR"] = (
    ds_summary
    .groupby(
        ["Environment", "Time_parsed"]
    )["DS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


ads_summary["ADS_CLR"] = (
    ads_summary
    .groupby(
        ["Environment", "Time_parsed"]
    )["ADS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


# ==========================================
# 5. Sort
# ==========================================

ds_summary = (
    ds_summary
    .sort_values(
        ["Environment", "DS", "Time_parsed"]
    )
    .reset_index(drop=True)
)

ads_summary = (
    ads_summary
    .sort_values(
        ["Environment", "ADS", "Time_parsed"]
    )
    .reset_index(drop=True)
)


# ==========================================
# 6. Calculate real observation changes
# ==========================================

print("Calculating changes and rates...")


# ---------- DS ----------

ds_summary["DS_delta"] = (
    ds_summary
    .groupby(
        ["Environment", "DS"]
    )["DS_TPM"]
    .diff()
)


ds_summary["DS_CLR_delta"] = (
    ds_summary
    .groupby(
        ["Environment", "DS"]
    )["DS_CLR"]
    .diff()
)


ds_summary["Time_interval_days"] = (
    ds_summary
    .groupby(
        ["Environment", "DS"]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# 平均变化速率
ds_summary["DS_rate"] = (
    ds_summary["DS_delta"]
    / ds_summary["Time_interval_days"]
)


ds_summary["DS_CLR_rate"] = (
    ds_summary["DS_CLR_delta"]
    / ds_summary["Time_interval_days"]
)


# ---------- ADS ----------

ads_summary["ADS_delta"] = (
    ads_summary
    .groupby(
        ["Environment", "ADS"]
    )["ADS_TPM"]
    .diff()
)


ads_summary["ADS_CLR_delta"] = (
    ads_summary
    .groupby(
        ["Environment", "ADS"]
    )["ADS_CLR"]
    .diff()
)


ads_summary["Time_interval_days"] = (
    ads_summary
    .groupby(
        ["Environment", "ADS"]
    )["Time_parsed"]
    .diff()
    .dt.days
)


ads_summary["ADS_rate"] = (
    ads_summary["ADS_delta"]
    / ads_summary["Time_interval_days"]
)


ads_summary["ADS_CLR_rate"] = (
    ads_summary["ADS_CLR_delta"]
    / ads_summary["Time_interval_days"]
)


# ==========================================
# 7. Save baseline data
# ==========================================

print("Saving baseline data...")

ds_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_baseline_with_rate.csv"
    ),
    index=False
)

ads_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ADS_baseline_with_rate.csv"
    ),
    index=False
)


# ==========================================
# 8. DS-ADS mapping
# ==========================================

ds_ads_mapping = {

    "Cas": "Anti_CRISPR",

    "RM": "Anti_RM",

    "CBASS": "Anti_CBASS",

    "Pycsar": "Anti_Pycsar",

    "Dnd": "Anti_Dnd",

    "Gabija": "Anti_Gabija",

    "Retron": "Anti_Retron",

    "Thoeris": "Anti_Thoeris"
}


# ==========================================
# 9. Linear interpolation function
# ==========================================

def interpolate_at_target(
    time_series,
    value_series,
    target_times,
    max_gap_days=60
):
    """
    在真实观测点之间，对指定 target_times 进行线性插值。

    注意：
    1. 不创建完整的daily time series
    2. 不进行区间外外推
    3. 如果相邻真实观测点间隔 > max_gap_days，则不插值
    """

    # 转换为numpy
    times = pd.to_datetime(time_series).reset_index(drop=True)
    values = pd.to_numeric(
        value_series,
        errors="coerce"
    ).reset_index(drop=True)

    # 去掉缺失
    valid = (
        times.notna()
        & values.notna()
    )

    times = times[valid]
    values = values[valid]

    # 排序
    order = np.argsort(times.values)

    times = times.iloc[order].reset_index(drop=True)
    values = values.iloc[order].reset_index(drop=True)

    results = []

    for target in pd.to_datetime(target_times):

        # --------------------------------------
        # 目标时间在真实观测范围之外
        # --------------------------------------

        if (
            target < times.iloc[0]
            or target > times.iloc[-1]
        ):
            results.append(np.nan)
            continue

        # --------------------------------------
        # 如果恰好存在真实观测
        # --------------------------------------

        exact_match = times == target

        if exact_match.any():

            idx = np.where(
                exact_match.values
            )[0][0]

            results.append(
                values.iloc[idx]
            )

            continue

        # --------------------------------------
        # 找到 target 左右两个真实观测点
        # --------------------------------------

        right_idx = times.searchsorted(
            target,
            side="right"
        )

        left_idx = right_idx - 1

        if (
            left_idx < 0
            or right_idx >= len(times)
        ):
            results.append(np.nan)
            continue

        t1 = times.iloc[left_idx]
        t2 = times.iloc[right_idx]

        y1 = values.iloc[left_idx]
        y2 = values.iloc[right_idx]

        # 实际时间间隔
        gap_days = (
            t2 - t1
        ).days

        # --------------------------------------
        # 插值区间超过限制
        # --------------------------------------

        if (
            gap_days <= 0
            or gap_days > max_gap_days
        ):
            results.append(np.nan)
            continue

        # --------------------------------------
        # 线性插值
        # --------------------------------------

        elapsed_days = (
            target - t1
        ).days

        ratio = (
            elapsed_days
            / gap_days
        )

        interpolated_value = (
            y1
            + (y2 - y1) * ratio
        )

        results.append(
            interpolated_value
        )

    return np.array(results)


# ==========================================
# 10. Correlation function
# ==========================================

def calculate_correlation(
    data,
    x_col,
    y_col,
    min_n=5
):

    temp = data.dropna(
        subset=[
            x_col,
            y_col
        ]
    )

    n = len(temp)

    if n < min_n:

        return (
            n,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    if (
        temp[x_col].nunique() < 2
        or
        temp[y_col].nunique() < 2
    ):

        return (
            n,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    spearman_r, spearman_p = spearmanr(
        temp[x_col],
        temp[y_col]
    )

    pearson_r, pearson_p = pearsonr(
        temp[x_col],
        temp[y_col]
    )

    return (
        n,
        spearman_r,
        spearman_p,
        pearson_r,
        pearson_p
    )


# ==========================================
# 11. Lagged analysis (Pooled Environments)
# ==========================================

print("\n========================================")
print("Running pooled lagged correlation analysis")
print("========================================")

results_pooled = []
lagged_data_all = []

# ==========================================
# 12. Analyze each DS-ADS pair
# ==========================================

for ds_name, ads_name in ds_ads_mapping.items():

    print(f"\nAnalyzing: {ds_name} vs {ads_name} (Pooled)")

    ds_temp = ds_summary[ds_summary["DS"] == ds_name].copy()
    ads_temp = ads_summary[ads_summary["ADS"] == ads_name].copy()

    # 遍历每一个lag天数
    for lag_days in TARGET_LAGS:

        # 用于存储跨环境合并后的配对数据
        x_clr_delta_pooled = []
        y_clr_delta_pooled = []
        x_clr_rate_pooled = []
        y_clr_rate_pooled = []

        # ==================================
        # 环境内部分别插值（时间序列必须在同环境对齐）
        # ==================================
        environments = sorted(set(ds_temp["Environment"]) | set(ads_temp["Environment"]))

        for env in environments:
            ds_env = ds_temp[ds_temp["Environment"] == env].sort_values("Time_parsed").reset_index(drop=True)
            ads_env = ads_temp[ads_temp["Environment"] == env].sort_values("Time_parsed").reset_index(drop=True)

            if len(ds_env) < 2 or len(ads_env) < 2:
                continue

            if INDEPENDENT_VARIABLE == "DS":
                x_time = ds_env["Time_parsed"]
                target_time = x_time + pd.to_timedelta(lag_days, unit="D")

                x_clr_delta = ds_env["DS_CLR_delta"].values
                x_clr_rate = ds_env["DS_CLR_rate"].values

                y_clr_delta = interpolate_at_target(ads_env["Time_parsed"], ads_env["ADS_CLR_delta"], target_time, INTERPOLATE_LIMIT)
                y_clr_rate = interpolate_at_target(ads_env["Time_parsed"], ads_env["ADS_CLR_rate"], target_time, INTERPOLATE_LIMIT)

            elif INDEPENDENT_VARIABLE == "ADS":
                x_time = ads_env["Time_parsed"]
                target_time = x_time + pd.to_timedelta(lag_days, unit="D")

                x_clr_delta = ads_env["ADS_CLR_delta"].values
                x_clr_rate = ads_env["ADS_CLR_rate"].values

                y_clr_delta = interpolate_at_target(ds_env["Time_parsed"], ds_env["DS_CLR_delta"], target_time, INTERPOLATE_LIMIT)
                y_clr_rate = interpolate_at_target(ds_env["Time_parsed"], ds_env["DS_CLR_rate"], target_time, INTERPOLATE_LIMIT)

            # 将当前环境配对好的有效数据追加到汇总池中
            x_clr_delta_pooled.extend(x_clr_delta)
            y_clr_delta_pooled.extend(y_clr_delta)
            x_clr_rate_pooled.extend(x_clr_rate)
            y_clr_rate_pooled.extend(y_clr_rate)

        # ==================================
        # 跨环境合并后，统一计算整体相关性
        # ==================================
        pooled_df = pd.DataFrame({
            "X_CLR_delta": x_clr_delta_pooled,
            "Y_CLR_delta": y_clr_delta_pooled,
            "X_CLR_rate": x_clr_rate_pooled,
            "Y_CLR_rate": y_clr_rate_pooled
        })

        (n_delta, spearman_r_delta, spearman_p_delta, _, _) = calculate_correlation(pooled_df, "X_CLR_delta", "Y_CLR_delta", MIN_N)
        (n_rate, spearman_r_rate, spearman_p_rate, _, _) = calculate_correlation(pooled_df, "X_CLR_rate", "Y_CLR_rate", MIN_N)

        results_pooled.append({
            "DS": ds_name,
            "ADS": ads_name,
            "Lag_Days": lag_days,
            "N_pooled": n_delta,
            "Spearman_r_CLR_delta": spearman_r_delta,
            "Spearman_p_CLR_delta": spearman_p_delta,
            "Spearman_r_CLR_rate": spearman_r_rate,
            "Spearman_p_CLR_rate": spearman_p_rate
        })


# ==========================================
# 13. Save pooled results
# ==========================================

print("\nSaving pooled correlation results...")
results_df = pd.DataFrame(results_pooled)
results_df.to_csv(os.path.join(OUTPUT_DIR, "Pooled_DS_ADS_lagged_correlation_results.csv"), index=False)


# ==========================================
# 14. Plot Faceted Plots with Significance Stars
# ==========================================

print("Generating faceted lag plots...")

def get_significance_stars(p_value):
    """根据 p-value 返回显著性星号"""
    if pd.isna(p_value): return ""
    if p_value < 0.001: return "***"
    if p_value < 0.01: return "**"
    if p_value < 0.05: return "*"
    return ""

def plot_faceted_lag_correlation(data, metric_r_col, metric_p_col, ylabel, filename):
    """绘制2x4的8分面图"""
    # 创建 2 行 4 列的画布 (共8个图，对应8对 DS-ADS)
    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, (ds_name, ads_name) in enumerate(ds_ads_mapping.items()):
        ax = axes[idx]

        # 提取这一对系统在不同 Lag 下的汇总结果
        subset = data[(data["DS"] == ds_name) & (data["ADS"] == ads_name)].sort_values("Lag_Days")

        if subset.empty or subset[metric_r_col].isna().all():
            ax.set_title(f"{ds_name} vs {ads_name}")
            continue

        # 绘制折线图
        ax.plot(subset["Lag_Days"], subset[metric_r_col], marker="o", linewidth=2, color="#2c7fb8", markersize=6)

        # 添加 y=0 的基准线
        ax.axhline(0, linestyle="--", linewidth=1.2, color="gray", alpha=0.7)

        # 标注显著性星号
        for _, row in subset.iterrows():
            x = row["Lag_Days"]
            y = row[metric_r_col]
            p = row[metric_p_col]

            if not pd.isna(y):
                stars = get_significance_stars(p)
                if stars:
                    # 将星号标记在数据点稍微偏上的位置
                    ax.text(x, y, stars, ha='center', va='bottom', color='red', fontsize=12, fontweight='bold')

        # 设置子图标题和坐标轴标签
        ax.set_title(f"{ds_name} vs {ads_name}", fontsize=12, fontweight='bold')
        ax.grid(True, linestyle=":", alpha=0.6)

        # 仅在底部和左侧显示坐标轴标签
        if idx >= 4:  # 第二行的图显示 X 轴
            ax.set_xlabel("Lag (Days)", fontsize=11)
        if idx % 4 == 0:  # 第一列的图显示 Y 轴
            ax.set_ylabel(ylabel, fontsize=11)

    # 调整布局间距
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches="tight")
    plt.close()


# 绘制 CLR Delta 分面图
plot_faceted_lag_correlation(
    data=results_df,
    metric_r_col="Spearman_r_CLR_delta",
    metric_p_col="Spearman_p_CLR_delta",
    ylabel="Spearman r (ΔCLR)",
    filename="Faceted_Lag_CLR_delta_Pooled.pdf"
)

# 绘制 CLR Rate 分面图
plot_faceted_lag_correlation(
    data=results_df,
    metric_r_col="Spearman_r_CLR_rate",
    metric_p_col="Spearman_p_CLR_rate",
    ylabel="Spearman r (CLR Rate)",
    filename="Faceted_Lag_CLR_rate_Pooled.pdf"
)

# ==========================================
# 15. Finish
# ==========================================

print("\n========================================")
print("Analysis and plotting finished successfully!")
print("========================================")
print(f"Results saved to:\n{OUTPUT_DIR}")